In [1]:
# =========================================
# BLOQUE 0: Importar librerías
# =========================================
import pandas as pd
import numpy as np

In [2]:
# =========================================
# BLOQUE 1: Rutas de entrada/salida (EDITA)
# =========================================
PATH_2009 = "/content/us_county_share_2G_2009.csv"   # <-- tu archivo 2009 (2G)
PATH_2011 = "/content/us_county_share_2G_2011.csv"   # <-- tu archivo 2011 (2G)
OUT_2010  = "/content/us_county_share_2G_2010.csv"   # <-- salida con promedio

In [3]:
# =========================================
# BLOQUE 2: Cargar y normalizar columnas
# =========================================
req_cols = {"county_fips", "year", "tech", "share_internet2G"}

df09 = pd.read_csv(PATH_2009)
df11 = pd.read_csv(PATH_2011)

# chequear columnas
assert req_cols.issubset(df09.columns), f"Faltan columnas en 2009: {req_cols - set(df09.columns)}"
assert req_cols.issubset(df11.columns), f"Faltan columnas en 2011: {req_cols - set(df11.columns)}"

# tipos
for df in (df09, df11):
    df["county_fips"] = df["county_fips"].astype(str).str.zfill(5)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["share_internet2G"] = pd.to_numeric(df["share_internet2G"], errors="coerce")

In [4]:
# =========================================
# BLOQUE 3: Validaciones simples
# =========================================
# unicidad por condado
assert df09["county_fips"].is_unique, "Duplicados en county_fips (2009)."
assert df11["county_fips"].is_unique, "Duplicados en county_fips (2011)."

# año correcto en cada archivo
assert set(df09["year"].dropna().unique()) == {2009}, "El CSV 2009 no tiene solo year=2009."
assert set(df11["year"].dropna().unique()) == {2011}, "El CSV 2011 no tiene solo year=2011."

In [5]:
# =========================================
# BLOQUE 4: Construir 2010 = promedio(2009, 2011)
# =========================================
base = df09[["county_fips", "share_internet2G"]].merge(
    df11[["county_fips", "share_internet2G"]].rename(columns={"share_internet2G": "share_internet2G_2011"}),
    on="county_fips",
    how="inner"
)

df2010 = pd.DataFrame({
    "county_fips": base["county_fips"],
    "year": 2010,
    "tech": "2G",
    "share_internet2G": (base["share_internet2G"] + base["share_internet2G_2011"]) / 2.0
})

# (opcional) acotar si es proporción [0,1]
# df2010["share_internet2G"] = df2010["share_internet2G"].clip(0, 1)

print("Filas 2010 creadas:", len(df2010))
df2010.head()

Filas 2010 creadas: 3108


,county_fips,year,tech,share_internet2G
0,01001,2010,2G,0.970319
1,01003,2010,2G,0.985825
2,01005,2010,2G,0.688761
3,01007,2010,2G,0.980128
4,01009,2010,2G,1.000000


In [6]:
# =========================================
# BLOQUE 5: Guardar SOLO 2010 (2G)
# =========================================
# asegurar orden exacto de columnas solicitado
df2010 = df2010[["county_fips", "year", "tech", "share_internet2G"]]
df2010.to_csv(OUT_2010, index=False)
print(f"Archivo 2010 guardado en: {OUT_2010}")

Archivo 2010 guardado en: /content/us_county_share_2G_2010.csv
